# Análisis exploratorio de incidencia delictiva en México

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [3]:
df = pd.read_csv('data/delitos.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/delitos.csv'

#### Ejercicio 1: Elige 3 estados de la república y grafica una serie de tiempo de la frecuencia abosluta de homicidios dolosos de enero 2015 a julio 2019 en estas tres entidades

In [ ]:
# ejercicio 1

# Filtrar homicidio doloso
df_homicidios = df[df['Subtipo de delito'] == 'Homicidio doloso']

# Elegir 3 estados
estados = ['Jalisco', 'Colima', 'Ciudad de México']

df_estados = df_homicidios[df_homicidios['Entidad'].isin(estados)]

# Crear fecha
df_estados['fecha'] = pd.to_datetime(
    df_estados['Año'].astype(str) + '-' + df_estados['Mes'] + '-01'
)

# Agrupar
df_group = df_estados.groupby(['Entidad', 'fecha'])['Total'].sum().reset_index()

# Graficar
for estado in estados:
    datos = df_group[df_group['Entidad'] == estado]
    plt.plot(datos['fecha'], datos['Total'], label=estado)

plt.legend()
plt.title("Homicidios dolosos por estado")
plt.xlabel("Fecha")
plt.ylabel("Casos")
plt.show()


#### Ejercicio 2: Contetas las siguientes  preguntas:
1. ¿Cuántos homicidios dolosos hubo en Colima en el 2018?
2. ¿Cuantos robos de vehículo automotor ha habido en el 2019?
3. Obten la suma de homicidos dolosos y feminidios en toda la República Mexicana en cada año.
4. ¿En qué mes y en qué municipio ha ocurrido el mayor número de feminicidios?
5. ¿En qué año y en qué estado ha ocurrido el mayor número de feminicidios?

In [ ]:
# 2.1
colima_2018 = df[
    (df['Entidad'] == 'Colima') &
    (df['Año'] == 2018) &
    (df['Subtipo de delito'] == 'Homicidio doloso')
]['Total'].sum()

print(colima_2018)

# 2.2
robos_2019 = df[
    (df['Año'] == 2019) &
    (df['Tipo de delito'] == 'Robo de vehículo automotor')
]['Total'].sum()

print(robos_2019)

# 2.3
df_filtrado = df[
    df['Subtipo de delito'].isin(['Homicidio doloso', 'Feminicidio'])
]

suma_por_anio = df_filtrado.groupby('Año')['Total'].sum()

print(suma_por_anio)

# 2.4
feminicidios = df[df['Subtipo de delito'] == 'Feminicidio']

max_fem = feminicidios.loc[feminicidios['Total'].idxmax()]

print(max_fem[['Mes', 'Municipio', 'Total']])

# 2.5
max_fem_estado = feminicidios.loc[feminicidios['Total'].idxmax()]

print(max_fem_estado[['Año', 'Entidad', 'Total']])

#### Ejercicio 3: Haz una gráfica de pastel de tipos de delito. Deberás crear una gráfica para cada año. Utilzia la función subplots de matplotlib

In [ ]:
# ejercicio 3

# Agrupar por año y tipo de delito
df_group = df.groupby(['Año', 'Tipo de delito'])['Total'].sum().reset_index()

# Obtener años únicos
anios = df_group['Año'].unique()

# Crear subplots
fig, axes = plt.subplots(len(anios), 1, figsize=(8, 4 * len(anios)))

# Si solo hay un año, convertir a lista
if len(anios) == 1:
    axes = [axes]

# Crear una gráfica por año
for i, anio in enumerate(anios):
    data_anio = df_group[df_group['Año'] == anio]
    
    axes[i].pie(
        data_anio['Total'],
        labels=data_anio['Tipo de delito'],
        autopct='%1.1f%%'
    )
    axes[i].set_title(f"Distribución de delitos en {anio}")

plt.tight_layout()
plt.show()

---
#### Calcula la tasa por 100,000 habitantes
##### Tasa por 100,000 habitantes
Mostrar el total de delitos en una entidad no nos sirve de mucho. Es mucho más útil calcular la tasa de incidencia delictiva por cada 100,000 habitantes

$$
tasa = \frac{delitos\space totales}{población} \times 100,000
$$

Esta tasa la podemos anualizar multiplicándola por un factor de 12
$$
tasa\space anualizada = tasa \times 12
$$

Población por entidad federativa según [la encuesta intercensal 2015](https://www.inegi.org.mx/programas/intercensal/2015/)

No tienes que descargar nada. Ya están los datos en la carpeta data

In [ ]:
pobs = pd.read_csv('data/poblacion_entidades_2015.csv', encoding='iso-8859-1', sep=";")
pobs = pobs[['Cve_Entidad', 'Entidad', 'Poblacion']]
pobs = pobs.rename(columns={'Cve_Entidad':'clave_entidad', 'Entidad':'entidad', 'Poblacion':'poblacion'})
pobs.head()

In [ ]:
# Calcular delitos totales por entidad
delitos_entidad = df.groupby('Entidad')['Total'].sum().reset_index()

# Unir delitos con población
tasa_df = delitos_entidad.merge(pobs, left_on='Entidad', right_on='entidad')

# Calcular tasa por 100,000 habitantes
tasa_df['tasa_100k'] = (tasa_df['Total'] / tasa_df['poblacion']) * 100000

# Calcular tasa anualizada
tasa_df['tasa_anualizada'] = tasa_df['tasa_100k'] * 12

# Mostrar resultados
tasa_df[['Entidad', 'Total', 'poblacion', 'tasa_100k', 'tasa_anualizada']]